# 02 - Seleccion Optima de K, Metricas de Evaluacion y Variantes Algoritmicas

## Curso de Machine Learning: Modelos No Supervisados

En los modelos supervisados (Regresion y Clasificacion), la seleccion de hiperparametros se realiza minimizando un error de prediccion objetivo ($MSE$, *Log-Loss*, $1 - F_1$) mediante validacion cruzada sobre datos etiquetados. En el agrupamiento no supervisado enfrentamos un desafio epistemologico mayor: **no existe $y_{test}$ contra el cual comparar nuestras predicciones**.

Ademas, no podemos simplemente elegir el hiperparametro $K$ que minimice la funcion objetivo de K-Means (la Inercia $J$), ya que la inercia disminuye monotonamente a medida que aumentamos $K$, llegando a ser exactamente $J = 0$ cuando $K = n$ (donde cada observacion forma su propio cluster individual, un analogo extremo al sobreajuste).

En este cuaderno estudiaremos las **metricas de validacion interna** que evaluan la calidad topologica de una particion, asi como las variantes de inicializacion (`k-means++`) y escalabilidad para Big Data (`MiniBatchKMeans`).

---

### Contenidos
1. El Problema de la Inercia Monotona y el Metodo del Codo (Elbow Method)
2. Coeficiente de Silueta (Silhouette Score):Formulacion y Analisis por Muestra
3. Indice de Calinski-Harabasz (Variance Ratio Criterion)
4. Indice de Davies-Bouldin
5. Diagnostico Visual Avanzado: Diagramas de Cuchillas de Silueta (Silhouette Plots)
6. Inicializacion Inteligente: K-Means++ vs. Aleatoria (`init` y `n_init`)
7. Escalabilidad para Grandes Volumenes de Datos: `MiniBatchKMeans`
8. Comparativa Integral de Metricas para Seleccionar $K^*$ en el Dataset Wine
9. Conclusiones


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.datasets import load_wine, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import (
    silhouette_score, 
    silhouette_samples, 
    calinski_harabasz_score, 
    davies_bouldin_score
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. El Problema de la Inercia Monotona y el Metodo del Codo

Recordemos que la inercia mide la varianza intra-cluster total:
$$WCSS(K) = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|_2^2$$

Al incrementar $K \rightarrow K+1$, siempre es posible subdividir un cluster existente y reducir $WCSS$. El **Metodo del Codo** (*Elbow Method*) consiste en graficar $WCSS(K)$ en funcion de $K$ e identificar el punto de inflexion donde la ganancia marginal por anadir un nuevo cluster cae abruptamente (pasamos de explicar estructura real a subdividir ruido interno).

Sin embargo, en datasets de alta dimensionalidad con variables correlacionadas, la curva de inercia suele ser suave y carecer de un codo pronunciado, por lo que requerimos metricas formales que ponderen simultaneamente **cohesion interna** y **separacion externa**.


## 2. Coeficiente de Silueta (Silhouette Score)

Propuesto por Peter Rousseeuw (1987), el **Coeficiente de Silueta** evalua que tan bien asignada esta cada observacion $i$ comparando su cercania a su propio cluster frente a su cercania al cluster vecino mas proximo.

Para cada muestra $x_i \in C_k$, definimos:
1. **Distancia media intra-cluster $a(i)$** (Cohesion): Distancia promedio entre $x_i$ y todos los demas puntos de su mismo cluster $C_k$:
   $$a(i) = \frac{1}{|C_k| - 1} \sum_{x_j \in C_k, j \neq i} \|x_i - x_j\|_2$$
2. **Distancia media al cluster vecino mas cercano $b(i)$** (Separacion): Minima distancia promedio entre $x_i$ y todos los puntos de cualquier otro cluster $C_m \neq C_k$:
   $$b(i) = \min_{m \neq k} \frac{1}{|C_m|} \sum_{x_j \in C_m} \|x_i - x_j\|_2$$

El **valor de silueta de la observacion $i$** se define como:
$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} \in [-1, +1]$$

### Interpretacion:
- **$s(i) \approx +1$**: Excelente asignacion ($a(i) \ll b(i)$). El punto esta muy compacto dentro de su grupo y lejos de los demas.
- **$s(i) \approx 0$**: Punto fronterizo ($a(i) \approx b(i)$). La observacion se encuentra exactamente sobre el hiperplano de Voronoi entre dos clusters.
- **$s(i) < 0$**: Asignacion erronea ($a(i) > b(i)$). El punto esta en promedio mas cerca de un cluster vecino que de los miembros de su propio cluster.


## 3. Indices de Calinski-Harabasz y Davies-Bouldin

Ademas de la Silueta (cuyo costo computacional es $O(n^2)$ por requerir la matriz de distancias por pares), existen dos indices rapidisimos ($O(n)$) basados en la dispersion respecto a los centroides:

### A. Indice de Calinski-Harabasz (CH - Criterio de Razon de Varianza)
Mide la relacion entre la dispersion inter-cluster ($BGSS$, *Between-Group Sum of Squares*) y la dispersion intra-cluster ($WGSS$, *Within-Group Sum of Squares*), penalizada por los grados de libertad:
$$CH(K) = \frac{BGSS / (K - 1)}{WGSS / (n - K)} \quad \text{(Mayor es mejor)}$$

### B. Indice de Davies-Bouldin (DB)
Para cada cluster $k$, identifica el "peor escenario" de solapamiento con otro cluster $j$, calculando la razon entre la suma de sus diametros internos ($\sigma_k + \sigma_j$) y la distancia entre sus centroides ($\|\mu_k - \mu_j\|_2$):
$$DB(K) = \frac{1}{K} \sum_{k=1}^{K} \max_{j \neq k} \left( \frac{\sigma_k + \sigma_j}{\|\mu_k - \mu_j\|_2} \right) \quad \text{(Menor es mejor, optimo en } 0\text{)}$$


In [ ]:
# Evaluacion multicriterio de K en el dataset Wine estandarizado
wine = load_wine(as_frame=True)
X_scaled = StandardScaler().fit_transform(wine.data)

rango_k = range(2, 11)
metricas_k = []

for k in rango_k:
    km = KMeans(n_clusters=k, init='k-means++', n_init=15, random_state=42)
    labels = km.fit_predict(X_scaled)
    
    metricas_k.append({
        'K': k,
        'Inercia': km.inertia_,
        'Silhouette': silhouette_score(X_scaled, labels),
        'Calinski_Harabasz': calinski_harabasz_score(X_scaled, labels),
        'Davies_Bouldin': davies_bouldin_score(X_scaled, labels)
    })

df_metricas = pd.DataFrame(metricas_k)
print("Tabla de Evaluacion Multicriterio para Seleccion de K:")
print(df_metricas.round(4).to_string(index=False))


In [ ]:
# Visualizacion del Panel de 4 Metricas de Validacion Interna
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Inercia (Codo)
axes[0, 0].plot(df_metricas['K'], df_metricas['Inercia'], 'o-', color='#1f77b4', linewidth=2)
axes[0, 0].axvline(3, color='red', linestyle='--', alpha=0.7, label='Codo Optimo (K=3)')
axes[0, 0].set_title('1. Metodo del Codo: Inercia (WCSS) - Minimizar', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Numero de Clusters (K)')
axes[0, 0].set_ylabel('Inercia')
axes[0, 0].legend()

# 2. Silhouette Score
axes[0, 1].plot(df_metricas['K'], df_metricas['Silhouette'], 's-', color='#2ca02c', linewidth=2)
axes[0, 1].axvline(3, color='red', linestyle='--', alpha=0.7, label='Maximo Global (K=3)')
axes[0, 1].set_title('2. Coeficiente de Silueta Medio - Maximizar', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Numero de Clusters (K)')
axes[0, 1].set_ylabel('Silhouette Score')
axes[0, 1].legend()

# 3. Calinski-Harabasz
axes[1, 0].plot(df_metricas['K'], df_metricas['Calinski_Harabasz'], '^-', color='#ff7f0e', linewidth=2)
axes[1, 0].axvline(3, color='red', linestyle='--', alpha=0.7, label='Maximo Global (K=3)')
axes[1, 0].set_title('3. Indice de Calinski-Harabasz - Maximizar', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Numero de Clusters (K)')
axes[1, 0].set_ylabel('CH Score')
axes[1, 0].legend()

# 4. Davies-Bouldin
axes[1, 1].plot(df_metricas['K'], df_metricas['Davies_Bouldin'], 'd-', color='#9467bd', linewidth=2)
axes[1, 1].axvline(3, color='red', linestyle='--', alpha=0.7, label='Minimo Global (K=3)')
axes[1, 1].set_title('4. Indice de Davies-Bouldin - Minimizar', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Numero de Clusters (K)')
axes[1, 1].set_ylabel('DB Score')
axes[1, 1].legend()

plt.suptitle('Consenso de Metricas Internas: Todas Identifican K=3 sin Usar Etiquetas', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 4. Diagnostico Visual Avanzado: Diagramas de Silueta por Muestra

El promedio global de Silueta resume la particion en un unico escalar, pero puede ocultar asimetrias graves (ej. dos clusters excelentes y un tercer cluster artificialmente fragmentado). 

Construyendo el **Diagrama de Cuchillas de Silueta** (*Silhouette Plot*) usando `silhouette_samples`, podemos inspeccionar:
1. **El grosor vertical de cada bloque**: Indica si los clusters tienen tamanos equilibrados.
2. **La extension horizontal de cada cuchilla**: Comprueba que todos los clusters superen la linea punteada roja del promedio global.
3. **La cola izquierda negativa**: Detecta muestras mal clasificadas ($s(i) < 0$).


In [ ]:
# Diagnostico mediante Diagramas de Cuchillas de Silueta para K = 2, 3, 4, 5
k_candidatos = [2, 3, 4, 5]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for idx, k in enumerate(k_candidatos):
    ax = axes[idx]
    km = KMeans(n_clusters=k, init='k-means++', n_init=15, random_state=42)
    cluster_labels = km.fit_predict(X_scaled)
    
    sil_avg = silhouette_score(X_scaled, cluster_labels)
    sample_silhouette_values = silhouette_samples(X_scaled, cluster_labels)
    
    y_lower = 10
    for i in range(k):
        ith_cluster_sv = sample_silhouette_values[cluster_labels == i]
        ith_cluster_sv.sort()
        
        size_cluster_i = ith_cluster_sv.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = cm.viridis(float(i) / k)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_sv,
                         facecolor=color, edgecolor=color, alpha=0.8)
        ax.text(-0.05, y_lower + 0.5 * size_cluster_i, f'C{i} (n={size_cluster_i})', fontsize=9, fontweight='bold')
        y_lower = y_upper + 10
        
    ax.axvline(x=sil_avg, color='red', linestyle='--', linewidth=2, label=f'Media: {sil_avg:.3f}')
    ax.axvline(x=0.0, color='black', linestyle='-', linewidth=0.8)
    ax.set_title(f'Diagrama de Silueta para K = {k}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Coeficiente de Silueta s(i)')
    ax.set_ylabel('Clusters y Muestras Ordenadas')
    ax.set_xlim([-0.15, 0.65])
    ax.set_yticks([])
    ax.legend(loc='upper right')

plt.suptitle('Inspeccion Estructural de Silueta: Equilibrio de Tamanos y Cohesion en K=3', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 5. Inicializacion Inteligente: K-Means++ vs. Aleatoria (`init` y `n_init`)

Como la funcion objetivo de K-Means no es convexa, una mala eleccion inicial de los centroides (por ejemplo, elegir por azar dos centroides dentro del mismo grupo denso) puede hacer que el algoritmo converja a un **optimo local deficiente**.

Para mitigar esto, Arthur y Vassilvitskii (2007) introdujeron **K-Means++**:
1. Se elige el primer centroide $\mu_1$ de manera uniformemente aleatoria entre los puntos de $X$.
2. Para cada punto $x_i$, se calcula su distancia al cuadrado $D(x_i)^2$ respecto al centroide mas cercano ya elegido.
3. Se selecciona el siguiente centroide $\mu_k$ muestreando de $X$ con una probabilidad proporcional a $D(x_i)^2$.

Esto garantiza que los centroides iniciales esten geometricamente dispersos antes de iniciar las iteraciones de Lloyd, logrando menor inercia final y requiriendo menos iteraciones para converger.


In [ ]:
# Experimento Monte Carlo: Estabilidad de K-Means++ vs Inicializacion Aleatoria (n_init=1)
inercia_random = []
inercia_plusplus = []
iter_random = []
iter_plusplus = []

for seed in range(100):
    km_rnd = KMeans(n_clusters=3, init='random', n_init=1, random_state=seed).fit(X_scaled)
    km_pp = KMeans(n_clusters=3, init='k-means++', n_init=1, random_state=seed).fit(X_scaled)
    
    inercia_random.append(km_rnd.inertia_)
    inercia_plusplus.append(km_pp.inertia_)
    iter_random.append(km_rnd.n_iter_)
    iter_plusplus.append(km_pp.n_iter_)

print("Comparativa en 100 Ejecuciones Independientes (n_init=1):")
print(f"  Random Init   -> Inercia Media: {np.mean(inercia_random):.2f} (Std: {np.std(inercia_random):.2f}) | Iteraciones Medias: {np.mean(iter_random):.1f}")
print(f"  K-Means++     -> Inercia Media: {np.mean(inercia_plusplus):.2f} (Std: {np.std(inercia_plusplus):.2f}) | Iteraciones Medias: {np.mean(iter_plusplus):.1f}")


## 6. Escalabilidad para Big Data: `MiniBatchKMeans`

Cuando el conjunto de datos contiene millones de observaciones ($n \gg 10^6$), calcular en cada iteracion la distancia de **todos** los puntos a todos los centroides resulta prohibitivo en memoria RAM y tiempo de CPU.

**`MiniBatchKMeans`** (Sculley, 2010) aplica la filosofia del Descenso de Gradiente Estocastico:
- En cada iteracion, extrae un lote aleatorio pequeno de tamano `batch_size` (ej. 1024 muestras) sin cargar todo el dataset.
- Asigna solo las muestras del mini-lote a sus centroides mas cercanos.
- Actualiza los centroides mediante un promedio movil convexo ponderado por el historico de muestras asignadas a cada centroide.

El resultado es una reduccion drastica del tiempo de computo a cambio de un ligero incremento marginal en la inercia final.


In [ ]:
# Benchmark de Escalabilidad: KMeans Estandar vs MiniBatchKMeans sobre 100,000 registros
X_big, _ = make_blobs(n_samples=100_000, n_features=20, centers=10, random_state=42)

t0 = time.perf_counter()
km_std = KMeans(n_clusters=10, init='k-means++', n_init=5, random_state=42).fit(X_big)
t_std = time.perf_counter() - t0

t0 = time.perf_counter()
km_mb = MiniBatchKMeans(n_clusters=10, init='k-means++', batch_size=1024, n_init=5, random_state=42).fit(X_big)
t_mb = time.perf_counter() - t0

diff_inercia_pct = ((km_mb.inertia_ - km_std.inertia_) / km_std.inertia_) * 100

print(f"Benchmark sobre N = 100,000 muestras y P = 20 dimensiones (K=10):")
print(f"  KMeans Estandar:   Tiempo = {t_std:.4f} seg | Inercia = {km_std.inertia_:,.1f}")
print(f"  MiniBatchKMeans:   Tiempo = {t_mb:.4f} seg | Inercia = {km_mb.inertia_:,.1f}")
print(f"  Aceleracion:       {t_std / t_mb:.1f}x mas rapido con solo +{diff_inercia_pct:.3f}% de diferencia en inercia")


## 7. Conclusiones

### Puntos Clave:
1. **Consenso de Metricas Internas**: Nunca dependas exclusivamente del grafico del codo. Combina el **Coeficiente de Silueta**, el **Indice de Calinski-Harabasz** y el **Indice de Davies-Bouldin** para encontrar un consenso robusto sobre $K^*$.
2. **Inspeccion por Muestra**: Los diagramas de cuchillas de Silueta permiten verificar visualmente que no existan clusters residuales desbalanceados ni puntos con silueta negativa.
3. **Inicializacion Robusta**: Utiliza siempre `init='k-means++'` junto con `n_init >= 10` para garantizar estabilidad frente a optimos locales.
4. **Escalabilidad**: Para pipelines de Big Data o flujos en streaming donde $n > 10^5$, `MiniBatchKMeans` ofrece ordenes de magnitud de aceleracion manteniendo la calidad geometrica de los centroides.

En el siguiente cuaderno, **03_Interpretacion_de_Perfiles_y_Explicabilidad_Clusters.ipynb**, aprenderemos como traducir estos vectores matematicos de centroides en perfiles de negocio accionables e interpretables.
